# Notebook 13 — Despliegue: FastAPI + Docker

**Semana:** 13  
**Bloque:** 6.1 — Producción  
**Tiempo estimado:** 4-5 horas

## Qué vas a construir

- Una API REST completa para DocBot con FastAPI
- Todos los endpoints necesarios: `/health`, `/ingest`, `/chat`, `/chat/stream`, `/memory`
- Un `Dockerfile` y `.dockerignore` correctos
- Tests de la API con `httpx` (el cliente HTTP de FastAPI)
- **Reto final:** añade autenticación por API key a todos los endpoints

## Prerrequisitos

Semana 12 completada. DocBot funciona. Docker instalado.

```bash
pip install fastapi uvicorn[standard] httpx python-multipart
docker --version   # debe devolver algo
```

---
## Celda 0 — Setup y navegación al proyecto

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

# El código de la API vive en proyectos/docbot-api/
API_DIR = Path("../proyectos/docbot-api").resolve()
API_DIR.mkdir(exist_ok=True)

print(f"Directorio de la API: {API_DIR}")

# Verifica que Docker está disponible
try:
    resultado = subprocess.run(["docker", "--version"], capture_output=True, text=True)
    print(f"Docker: {resultado.stdout.strip()}")
except FileNotFoundError:
    print("Docker no encontrado. Instálalo desde https://docs.docker.com/get-docker/")
    print("Puedes continuar el notebook hasta la celda de Docker.")

---
## Celda 1 — Los modelos Pydantic de la API

Define los contratos de request y response antes de escribir los endpoints.
Pydantic valida automáticamente que los datos recibidos tienen el formato correcto.

In [ ]:
# Escribe los modelos en el directorio de la API
MODELOS_PY = '''
"""models.py — Contratos Pydantic de la API."""

from pydantic import BaseModel, Field
from typing import Optional


# ── Requests ──────────────────────────────────────────────────────────────────

class ChatRequest(BaseModel):
    pregunta:   str           = Field(..., min_length=1, max_length=2000,
                                       description="Pregunta del usuario")
    user_id:    str           = Field(default="anon", max_length=64,
                                       description="ID del usuario")
    session_id: Optional[str] = Field(default=None, description="ID de sesión existente")


class IngestRequest(BaseModel):
    fuentes:   list[dict]    = Field(..., description="Lista de fuentes a indexar")
    coleccion: str           = Field(default="rag_docs", description="Nombre de la colección")

    class Config:
        json_schema_extra = {
            "example": {
                "fuentes": [
                    {"tipo": "wikipedia", "valor": "FastAPI",
                     "titulo": "FastAPI", "categoria": "framework"}
                ]
            }
        }


# ── Responses ─────────────────────────────────────────────────────────────────

class ChatResponse(BaseModel):
    respuesta:    str
    fuentes:      list[str]
    n_chunks:     int
    respondida:   bool
    latencia_ms:  float
    session_id:   str
    faithfulness: Optional[float] = None


class IngestResponse(BaseModel):
    fuentes_ok:      int
    fuentes_error:   int
    chunks_totales:  int
    n_docs_total:    int


class HealthResponse(BaseModel):
    status:    str          # "ok" | "degraded"
    n_docs:    int
    version:   str = "1.0.0"


class MetricsResponse(BaseModel):
    user_id:             str
    n_total:             int
    n_respondidas:       int
    tasa_respuesta:      float
    faithfulness_media:  Optional[float]
    context_rel_media:   Optional[float]


class ErrorResponse(BaseModel):
    error:   str
    detalle: Optional[str] = None
'''

(API_DIR / "models.py").write_text(MODELOS_PY)
print("models.py creado.")

# ── Test de los modelos ───────────────────────────────────────────────────────
sys.path.insert(0, str(API_DIR))
os.chdir(API_DIR)

# Instala FastAPI si no está
try:
    from models import ChatRequest, ChatResponse, IngestRequest, HealthResponse
    print("Modelos importados correctamente.")
except ImportError:
    print("Instalando fastapi...")
    subprocess.run([sys.executable, "-m", "pip", "install", "fastapi", "uvicorn[standard]", "httpx", "-q"])
    from models import ChatRequest, ChatResponse, IngestRequest, HealthResponse

# Prueba de validación
req_valido = ChatRequest(pregunta="What is FastAPI?", user_id="raffaele")
print(f"\nRequest válido: {req_valido.model_dump()}")

try:
    req_invalido = ChatRequest(pregunta="")  # min_length=1 → debe fallar
except Exception as e:
    print(f"Request inválido (correcto): {type(e).__name__}")

---
## Celda 2 — La aplicación FastAPI completa

Escribe el archivo `api.py` con todos los endpoints.
Después arranca el servidor y prueba cada endpoint manualmente.

In [ ]:
API_PY = '''
"""
api.py — API REST de DocBot.

Arrancar con:
    uvicorn api:app --reload --port 8000
"""

import asyncio
import os
import sys
import threading
from contextlib import asynccontextmanager
from pathlib import Path
from typing import AsyncGenerator

from fastapi import FastAPI, HTTPException, Depends, Header
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv

load_dotenv()

# Añade el directorio de DocBot al path
sys.path.insert(0, str(Path(__file__).parent.parent / "docbot"))
from docbot import DocBot, RespuestaDocBot

from models import (
    ChatRequest, ChatResponse,
    IngestRequest, IngestResponse,
    HealthResponse, MetricsResponse, ErrorResponse
)


# ── Pool de bots por usuario ──────────────────────────────────────────────────

_bots: dict[str, DocBot] = {}
_lock = threading.Lock()

def obtener_bot(user_id: str) -> DocBot:
    """Devuelve el bot del usuario, creándolo si no existe."""
    with _lock:
        if user_id not in _bots:
            _bots[user_id] = DocBot(user_id=user_id)
        return _bots[user_id]


# ── Lifespan: acciones al arrancar/parar ──────────────────────────────────────

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Al arrancar: warm-up (instancia un bot de prueba)
    print("DocBot API arrancando...")
    if not os.getenv("OPENAI_API_KEY"):
        print("ADVERTENCIA: OPENAI_API_KEY no configurada")
    yield
    # Al parar: cierra las sesiones abiertas
    print("Cerrando sesiones...")
    for bot in _bots.values():
        try:
            bot.cerrar_sesion()
        except Exception:
            pass


# ── Aplicación ────────────────────────────────────────────────────────────────

app = FastAPI(
    title="DocBot API",
    description="RAG chatbot con memoria. Indexa documentos y responde preguntas.",
    version="1.0.0",
    lifespan=lifespan
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],   # En producción: lista explícita de dominios
    allow_methods=["*"],
    allow_headers=["*"],
)


# ── Endpoints ─────────────────────────────────────────────────────────────────

@app.get("/health", response_model=HealthResponse, tags=["sistema"])
async def health():
    """Health check. Los sistemas de orquestación llaman a este endpoint para
    verificar que el servicio está vivo antes de enviarle tráfico."""
    bot = obtener_bot("_health")
    return HealthResponse(status="ok", n_docs=bot.n_docs)


@app.post("/ingest", response_model=IngestResponse, tags=["documentos"])
async def ingest(req: IngestRequest):
    """Indexa documentos en el corpus. Soporta Wikipedia, Markdown y texto plano."""
    bot = obtener_bot("_ingest")   # Bot compartido para ingesta
    try:
        stats = await asyncio.to_thread(bot.ingestar, req.fuentes)
        return IngestResponse(
            fuentes_ok=stats["fuentes_ok"],
            fuentes_error=stats["fuentes_error"],
            chunks_totales=stats["chunks_totales"],
            n_docs_total=bot.n_docs
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/chat", response_model=ChatResponse, tags=["chat"])
async def chat(req: ChatRequest):
    """Responde una pregunta usando el corpus indexado."""
    bot = obtener_bot(req.user_id)
    try:
        # asyncio.to_thread: ejecuta la función síncrona sin bloquear el event loop
        resultado: RespuestaDocBot = await asyncio.to_thread(
            bot.responder, req.pregunta
        )
        return ChatResponse(
            respuesta=resultado.respuesta,
            fuentes=resultado.fuentes,
            n_chunks=resultado.n_chunks,
            respondida=resultado.respondida,
            latencia_ms=resultado.latencia_ms,
            session_id=bot.session_id,
            faithfulness=resultado.faithfulness
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/chat/stream", tags=["chat"])
async def chat_stream(req: ChatRequest):
    """
    Responde con Server-Sent Events (SSE).
    El cliente recibe los tokens del LLM en tiempo real.

    Formato de cada evento:
      data: {"token": "..."}\n\n
      data: {"done": true, "fuentes": [...]}\n\n
    """
    import json as _json
    from openai import OpenAI

    cliente_stream = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    async def generar_tokens() -> AsyncGenerator[str, None]:
        try:
            # Recupera el contexto RAG (síncrono, en thread separado)
            bot = obtener_bot(req.user_id)

            # Hacemos retrieval primero (síncrono)
            from docbot.retrieval import recuperar, reescribir_query, construir_prompt, UMBRAL_RELEVANCIA
            import chromadb
            chroma   = chromadb.PersistentClient(path="./chroma_db")
            coleccion = chroma.get_collection("rag_docs")

            query    = await asyncio.to_thread(reescribir_query, req.pregunta, cliente_stream)
            chunks   = await asyncio.to_thread(recuperar, query, coleccion, cliente_stream, 4)
            relevantes = [c for c in chunks if c.similitud >= UMBRAL_RELEVANCIA]

            if not relevantes:
                yield f"data: {_json.dumps({'token': "I don\'t have enough information to answer this."})}\n\n"
                yield f"data: {_json.dumps({'done': True, 'fuentes': []})}\n\n"
                return

            prompt  = construir_prompt(req.pregunta, relevantes)
            fuentes = list(dict.fromkeys(c.fuente for c in relevantes))

            # Streaming del LLM
            stream = cliente_stream.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "Answer using ONLY the provided context. Cite sources."},
                    {"role": "user",   "content": prompt}
                ],
                stream=True, temperature=0.1, max_tokens=600
            )
            for chunk in stream:
                delta = chunk.choices[0].delta.content
                if delta:
                    yield f"data: {_json.dumps({'token': delta})}\n\n"

            yield f"data: {_json.dumps({'done': True, 'fuentes': fuentes})}\n\n"

        except Exception as e:
            yield f"data: {_json.dumps({'error': str(e)})}\n\n"

    return StreamingResponse(generar_tokens(), media_type="text/event-stream")


@app.get("/metrics/{user_id}", response_model=MetricsResponse, tags=["evaluacion"])
async def get_metrics(user_id: str, n_ultimos: int = 10):
    """Devuelve las métricas de calidad de la sesión del usuario."""
    bot = obtener_bot(user_id)
    try:
        metricas = await asyncio.to_thread(bot.evaluar_sesion, n_ultimos)
        return MetricsResponse(
            user_id=user_id,
            n_total=metricas["n_total"],
            n_respondidas=metricas["n_respondidas"],
            tasa_respuesta=metricas["tasa_respuesta"],
            faithfulness_media=metricas.get("faithfulness_media"),
            context_rel_media=metricas.get("context_rel_media")
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.delete("/memory/{user_id}", tags=["memoria"])
async def delete_memory(user_id: str):
    """Borra toda la memoria del usuario (sesiones + memoria semántica)."""
    if user_id in _bots:
        with _lock:
            bot = _bots.pop(user_id, None)
        if bot:
            await asyncio.to_thread(bot.olvidar)
    return {"message": f"Memoria de '{user_id}' borrada correctamente."}
'''

(API_DIR / "api.py").write_text(API_PY)
print("api.py creado.")
print("\nPara arrancarlo manualmente:")
print(f"  cd {API_DIR}")
print("  uvicorn api:app --reload --port 8000")
print("\nDocs automáticas en: http://localhost:8000/docs")

---
## Celda 3 — Tests de la API con httpx

FastAPI incluye `TestClient` (basado en httpx) que permite probar
los endpoints sin arrancar un servidor real. Escribe tests para todos los endpoints.

In [ ]:
# IMPORTANTE: este bloque asume que DocBot y sus dependencias están instaladas
# y que existe una colección 'rag_docs' en ./chroma_db con al menos algunos chunks.
# Si no tienes la colección, ejecuta primero el notebook 12 para crearla.

# Crea el archivo de tests
TESTS_PY = '''
"""
test_api.py — Tests de integración para la API de DocBot.

Ejecutar con:
    pytest test_api.py -v
"""

import pytest
from fastapi.testclient import TestClient
from api import app

client = TestClient(app)


# ── Tests de health ───────────────────────────────────────────────────────────

def test_health_devuelve_ok():
    r = client.get("/health")
    assert r.status_code == 200
    datos = r.json()
    assert datos["status"] == "ok"
    assert "n_docs" in datos
    assert "version" in datos


# ── Tests de ingesta ──────────────────────────────────────────────────────────

def test_ingest_wikipedia():
    r = client.post("/ingest", json={
        "fuentes": [
            {"tipo": "wikipedia", "valor": "Python (programming language)",
             "titulo": "Python", "categoria": "lenguaje"}
        ]
    })
    assert r.status_code == 200
    datos = r.json()
    assert datos["fuentes_ok"] == 1
    assert datos["chunks_totales"] > 0


def test_ingest_fuente_invalida():
    """Fuente con tipo desconocido: debe procesarse sin error 500 (falla graciosamente)."""
    r = client.post("/ingest", json={
        "fuentes": [{"tipo": "pdf_inexistente", "valor": "nada.pdf", "titulo": "test"}]
    })
    assert r.status_code == 200
    datos = r.json()
    assert datos["fuentes_error"] >= 1   # falló, pero no lanzó 500


# ── Tests de chat ─────────────────────────────────────────────────────────────

def test_chat_pregunta_dentro_corpus():
    r = client.post("/chat", json={
        "pregunta": "What is FastAPI?",
        "user_id":  "test_user"
    })
    assert r.status_code == 200
    datos = r.json()
    assert "respuesta" in datos
    assert "session_id" in datos
    assert isinstance(datos["respondida"], bool)
    assert datos["latencia_ms"] > 0


def test_chat_pregunta_fuera_corpus():
    r = client.post("/chat", json={
        "pregunta": "What is the current stock price of Apple?",
        "user_id":  "test_user"
    })
    assert r.status_code == 200
    datos = r.json()
    # Puede responder o no, pero no debe devolver 500
    assert "respuesta" in datos


def test_chat_pregunta_vacia_falla_validacion():
    r = client.post("/chat", json={"pregunta": "", "user_id": "test_user"})
    assert r.status_code == 422   # Unprocessable Entity — Pydantic rechaza min_length=1


def test_chat_session_id_devuelto():
    """Dos requests del mismo usuario deben devolver el mismo session_id."""
    r1 = client.post("/chat", json={"pregunta": "What is Docker?", "user_id": "user_persistencia"})
    r2 = client.post("/chat", json={"pregunta": "And Kubernetes?",  "user_id": "user_persistencia"})
    assert r1.status_code == 200
    assert r2.status_code == 200
    # Mismo usuario → misma sesión
    assert r1.json()["session_id"] == r2.json()["session_id"]


# ── Tests de memoria ──────────────────────────────────────────────────────────

def test_delete_memory():
    # Crea algo de actividad
    client.post("/chat", json={"pregunta": "Hello", "user_id": "borrar_test"})
    # Borra la memoria
    r = client.delete("/memory/borrar_test")
    assert r.status_code == 200
    assert "borrado" in r.json()["message"].lower() or "borrada" in r.json()["message"].lower()


# ── Tests de estructura de respuesta ─────────────────────────────────────────

def test_chat_response_tiene_todos_los_campos():
    r = client.post("/chat", json={"pregunta": "What is JWT?", "user_id": "campos_test"})
    assert r.status_code == 200
    datos = r.json()
    campos_requeridos = ["respuesta", "fuentes", "n_chunks", "respondida", "latencia_ms", "session_id"]
    for campo in campos_requeridos:
        assert campo in datos, f"Campo ausente: {campo}"
'''

(API_DIR / "test_api.py").write_text(TESTS_PY)
print("test_api.py creado.")

# ── Ejecuta los tests (requiere que la API y DocBot estén instalados) ─────────
print("\nPara ejecutar los tests:")
print(f"  cd {API_DIR}")
print("  pip install pytest")
print("  pytest test_api.py -v")

# Intento de ejecución directa (puede fallar si faltan dependencias)
try:
    resultado = subprocess.run(
        [sys.executable, "-m", "pytest", "test_api.py", "-v", "--tb=short"],
        capture_output=True, text=True, cwd=API_DIR
    )
    print("\n" + resultado.stdout[-3000:] if len(resultado.stdout) > 3000 else "\n" + resultado.stdout)
    if resultado.returncode != 0:
        print("STDERR:", resultado.stderr[-500:])
except Exception as e:
    print(f"No se pudo ejecutar pytest automáticamente: {e}")
    print("Ejecuta los tests manualmente desde la terminal.")

---
## Celda 4 — Dockerfile y docker-compose

Empaqueta la API en un contenedor Docker.
Después constrúyelo y arráncalo localmente para verificar que funciona.

In [ ]:
# ── requirements.txt ──────────────────────────────────────────────────────────
REQUIREMENTS = """fastapi>=0.111.0
uvicorn[standard]>=0.30.0
openai>=1.30.0
anthropic>=0.25.0
chromadb>=0.5.0
langchain-text-splitters>=0.2.0
python-dotenv>=1.0.0
requests>=2.31.0
httpx>=0.27.0
pytest>=8.0.0
"""

# ── Dockerfile ────────────────────────────────────────────────────────────────
DOCKERFILE = """# ── Imagen base: Python 3.11 mínima (~150MB vs ~900MB full)
FROM python:3.11-slim

# Metadatos
LABEL maintainer="raffaele@dihdatalife.com"
LABEL version="1.0.0"
LABEL description="DocBot RAG API"

WORKDIR /app

# ── Instala dependencias primero (se cachea si requirements.txt no cambia)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# ── Copia el código del paquete docbot
COPY ../docbot ./docbot

# ── Copia la API
COPY api.py models.py ./

# ── Crea directorios de datos que se montarán como volúmenes
RUN mkdir -p chroma_db sesiones evaluaciones

# ── Puerto de la aplicación
EXPOSE 8000

# ── Variable de entorno para que Chroma sepa dónde guardar datos
ENV CHROMA_DB_PATH=/app/chroma_db

# ── Health check: Docker comprueba que el contenedor está vivo cada 30s
HEALTHCHECK --interval=30s --timeout=10s --start-period=15s --retries=3 \\
  CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"

# ── Arranque: 1 worker (aumenta con --workers N en producción multi-core)
CMD ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"]
"""

# ── .dockerignore ─────────────────────────────────────────────────────────────
DOCKERIGNORE = """.env
.env.*
chroma_db/
sesiones/
evaluaciones/
__pycache__/
*.pyc
*.pyo
.git/
.gitignore
*.ipynb
notebooks/
README.md
test_*.py
"""

# ── docker-compose.yml (para desarrollo local) ────────────────────────────────
COMPOSE = """version: '3.8'

services:
  docbot-api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - OPENAI_API_KEY=${OPENAI_API_KEY}
    volumes:
      # Datos persistentes FUERA del contenedor
      - ./chroma_db:/app/chroma_db
      - ./sesiones:/app/sesiones
      - ./evaluaciones:/app/evaluaciones
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "python", "-c",
             "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"]
      interval: 30s
      timeout: 10s
      retries: 3
"""

# Escribe todos los archivos
(API_DIR / "requirements.txt").write_text(REQUIREMENTS)
(API_DIR / "Dockerfile").write_text(DOCKERFILE)
(API_DIR / ".dockerignore").write_text(DOCKERIGNORE)
(API_DIR / "docker-compose.yml").write_text(COMPOSE)

print("Archivos de Docker creados:")
for f in ["requirements.txt", "Dockerfile", ".dockerignore", "docker-compose.yml"]:
    print(f"  {API_DIR / f}")

# ── Muestra el Dockerfile formateado ─────────────────────────────────────────
print("\n" + "=" * 55)
print("DOCKERFILE")
print("=" * 55)
print(DOCKERFILE)

In [ ]:
# ── Instrucciones para construir y correr el contenedor ───────────────────────
print("Para construir y arrancar el contenedor:")
print()
print(f"  cd {API_DIR}")
print("  docker build -t docbot-api:1.0 .")
print()
print("  # Opción A: docker run directo")
print("  docker run -p 8000:8000 \\")
print("    -e OPENAI_API_KEY=$OPENAI_API_KEY \\")
print("    -v $(pwd)/chroma_db:/app/chroma_db \\")
print("    -v $(pwd)/sesiones:/app/sesiones \\")
print("    docbot-api:1.0")
print()
print("  # Opción B: docker-compose (más cómodo)")
print("  docker-compose up")
print()
print("Verifica que funciona:")
print("  curl http://localhost:8000/health")
print("  # Debe devolver: {\"status\": \"ok\", ...}")
print()
print("Docs interactivas: http://localhost:8000/docs")

# Intento de build automático (puede tardar varios minutos la primera vez)
HACER_BUILD = False   # Cambia a True para ejecutar el build automáticamente
if HACER_BUILD:
    print("\nConstruyendo imagen Docker (puede tardar ~3 minutos)...")
    result = subprocess.run(
        ["docker", "build", "-t", "docbot-api:1.0", "."],
        capture_output=True, text=True, cwd=API_DIR
    )
    print(result.stdout[-2000:])
    if result.returncode == 0:
        print("✓ Imagen construida correctamente.")
    else:
        print("✗ Error en el build:")
        print(result.stderr[-1000:])

---
## Reto final — Autenticación por API key

Una API de producción no puede estar abierta a cualquiera.
Añade autenticación por API key usando una `Dependency` de FastAPI.

Lee el esqueleto y completa los `# TODO`.

In [ ]:
AUTH_PY = '''
"""
auth.py — Autenticación por API key para la API de DocBot.

Uso en los endpoints:
    from auth import verificar_api_key

    @app.post("/chat", dependencies=[Depends(verificar_api_key)])
    async def chat(req: ChatRequest): ...
"""

import os
import secrets
from fastapi import HTTPException, Security, status
from fastapi.security import APIKeyHeader

# El header donde el cliente envía la API key
# Ejemplo: curl -H "X-API-Key: mi-clave" http://localhost:8000/chat
api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)


def _cargar_claves_validas() -> set[str]:
    """
    Carga las API keys válidas desde variables de entorno.

    TODO: implementa esta función.
    Lee la variable de entorno API_KEYS (separadas por coma) y devuelve un set.
    Si la variable no existe, genera una clave aleatoria, la imprime en el log,
    y la devuelve como único elemento del set.

    Ejemplo de variable de entorno:
      API_KEYS=clave1,clave2,clave3
    """
    # TODO
    raise NotImplementedError


# Carga las claves al importar el módulo (una vez al arrancar)
_CLAVES_VALIDAS = _cargar_claves_validas()


async def verificar_api_key(api_key: str = Security(api_key_header)) -> str:
    """
    Dependencia FastAPI. Verifica que el request incluye una API key válida.

    TODO: implementa la verificación.
    - Si api_key es None o vacío: lanza HTTPException 401 (no autenticado)
    - Si api_key no está en _CLAVES_VALIDAS: lanza HTTPException 403 (prohibido)
    - Si es válida: devuelve la api_key

    Usa secrets.compare_digest en lugar de == para comparar strings
    (evita timing attacks).
    """
    # TODO
    raise NotImplementedError


def generar_nueva_clave() -> str:
    """
    Genera una API key segura de 32 bytes en formato hex.
    Útil para crear claves para nuevos clientes.
    """
    return secrets.token_hex(32)
'''

(API_DIR / "auth.py").write_text(AUTH_PY)
print("auth.py creado (con TODOs por implementar).")

# ── Test de la autenticación ──────────────────────────────────────────────────
AUTH_TEST = '''
"""
test_auth.py — Tests de autenticación.
Descomenta cuando hayas implementado auth.py.
"""

import os
import pytest
from fastapi.testclient import TestClient

# Configura una clave de test antes de importar la app
os.environ["API_KEYS"] = "test-key-12345"

# from api_con_auth import app   # versión de api.py con auth aplicada
# client = TestClient(app)

# def test_sin_api_key_devuelve_401():
#     r = client.post("/chat", json={"pregunta": "test"})
#     assert r.status_code == 401

# def test_clave_invalida_devuelve_403():
#     r = client.post("/chat",
#         json={"pregunta": "test"},
#         headers={"X-API-Key": "clave-incorrecta"}
#     )
#     assert r.status_code == 403

# def test_clave_valida_devuelve_200():
#     r = client.post("/chat",
#         json={"pregunta": "What is FastAPI?"},
#         headers={"X-API-Key": "test-key-12345"}
#     )
#     assert r.status_code == 200
'''

(API_DIR / "test_auth.py").write_text(AUTH_TEST)
print("test_auth.py creado.")

# ── Pistas ────────────────────────────────────────────────────────────────────
print("\nPistas para implementar auth.py:")
print("""  _cargar_claves_validas():
    raw = os.getenv("API_KEYS", "")
    if raw:
        return {k.strip() for k in raw.split(",") if k.strip()}
    else:
        nueva = secrets.token_hex(32)
        print(f"API_KEYS no configurada. Clave temporal: {nueva}")
        return {nueva}

  verificar_api_key():
    if not api_key:
        raise HTTPException(status_code=401, detail="API key requerida")
    # Compara contra cada clave válida con compare_digest
    for clave_valida in _CLAVES_VALIDAS:
        if secrets.compare_digest(api_key, clave_valida):
            return api_key
    raise HTTPException(status_code=403, detail="API key inválida")
""")

### Extensiones opcionales

1. **Rate limiting:** añade un límite de requests por usuario por minuto usando `slowapi` (wrapper de Flask-Limiter para FastAPI).

2. **Logging estructurado:** usa `structlog` para que cada request genere un log JSON con `user_id`, `latencia_ms`, `n_chunks` y `respondida`. En producción estos logs van a Datadog, Grafana Loki o similar.

3. **Deploy en Railway:** [Railway](https://railway.app) despliega contenedores Docker automáticamente desde GitHub. Puedes tener tu API pública con URL propia en menos de 10 minutos y plan gratuito.

In [ ]:
# Resumen de todos los archivos creados esta semana
print("Archivos generados en esta semana:")
print()
for f in sorted(API_DIR.rglob("*")):
    if f.is_file() and not any(p in str(f) for p in ["__pycache__", ".pyc"]):
        print(f"  {f.relative_to(API_DIR.parent)}")

---
## Autoevaluación de la semana

1. **¿Qué funcionó bien?**  
   ...

2. **¿Qué no entendiste del todo?**  
   ...

3. **¿Qué cambiarías la semana que viene?**  
   ...

4. **Tiempo real dedicado:**  
   ...

---
## Próxima semana

La semana 14 completa el bloque de producción:
**despliegue en la nube** (Railway o Render), variables de entorno en producción,
monitorización básica y el ciclo completo desde `git push` hasta URL pública.

---
*Notebook 13 — Semana 13 del Plan de Formación AI Engineering*